# Exercise: Weather Station Data Analysis

You have temperature readings (in **Kelvin**) from 4 weather stations collected over 6 measurements.
Your task is to analyse this data using PyTorch tensor operations.

**Topics covered:** tensor creation, arithmetic, broadcasting, reduction operators, math functions

In [1]:
import torch
import math

print(f"PyTorch version: {torch.__version__}")

PyTorch version: 1.13.1


---
## Task 1: Create the data (tensor creation)

1. Set the random seed to `42` for reproducibility.
2. Generate a `(4, 6)` tensor of random temperatures in **Kelvin** in the range **[253, 313]** (i.e. roughly -20°C to 40°C - a range of 60°C).  
   *Hint:* `torch.rand()` gives values in [0, 1). Scale and shift to get the desired range.
3. Create a 1D tensor `altitudes` with station altitudes in metres: `[150, 420, 890, 1200]` (use `dtype=torch.float32`).
4. Print the shape and dtype of both tensors.

In [ ]:
# 1. Reproducibility
??

# 2. Random temperatures in Kelvin [253, 313]
# Hint:* `torch.rand()` gives values in [0, 1). Scale by 60 and shift to start from 253.
temps_k = ??

# 3. Station altitudes
# Create a 1D tensor `altitudes` with station altitudes in metres: `[150, 420, 890, 1200]` (use `dtype=torch.float32`).
altitudes = ??

# 4. Inspect
print(f"temps_k:   shape={temps_k.shape}  dtype={temps_k.dtype}")
print(f"altitudes: shape={altitudes.shape}  dtype={altitudes.dtype}")
print()
print("Temperature readings (K):")
print(temps_k)


### Solution

In [ ]:
# Solution Task 1

'''
# 1. Reproducibility
torch.manual_seed(42)

# 2. Random temperatures in Kelvin [253, 313]
temps_k = torch.rand(4, 6) * 60 + 253  # rand in [0,1) -> [253, 313)

# 3. Station altitudes
altitudes = torch.tensor([150, 420, 890, 1200], dtype=torch.float32)

# 4. Inspect
print(f"temps_k:   shape={temps_k.shape}  dtype={temps_k.dtype}")
print(f"altitudes: shape={altitudes.shape}  dtype={altitudes.dtype}")
print()
print("Temperature readings (K):")
print(temps_k)
'''

---
## Task 2: Temperature conversion & altitude correction (arithmetic + broadcasting)

1. Convert all temperatures from **Kelvin to Celsius**: $C = K - 273.15$
2. Apply an altitude-based temperature correction — higher stations are colder.  
   Subtract `0.0065 * altitude` from each station’s temperatures.  
   *Hint:* reshape `altitudes` to `(4, 1)` so it broadcasts over the 6 measurements.
3. Print the corrected temperature matrix and verify its shape is `(4, 6)`.

In [ ]:
# 1. Kelvin -> Celsius
temps_c = ??

# 2. Altitude correction: broadcast (4,1) over (4,6)
# Calculate correction for each altitude `0.0065 * altitude`
# *Hint:* reshape `altitudes` to `(4, 1)` so it broadcasts over the 6 measurements
correction = ??
# Apply the correction to each station’s temperatures
temps_corrected = ??

# 3. Verify
print(f"Shape: {temps_corrected.shape}")
print()
print("Corrected temperatures (°C):")
print(temps_corrected)

### Solution

In [ ]:
# Solution Task 2

'''
# 1. Kelvin -> Celsius
temps_c = temps_k - 273.15

# 2. Altitude correction: broadcast (4,1) over (4,6)
correction = 0.0065 * altitudes.reshape(4, 1)  # shape (4, 1)
temps_corrected = temps_c - correction          # shape (4, 6)

# 3. Verify
print(f"Shape: {temps_corrected.shape}")
print()
print("Corrected temperatures (°C):")
print(temps_corrected)
'''

---
## Task 3: Statistics (reduction operators)

Using the corrected Celsius temperatures:

1. Compute the **mean** and **standard deviation** per station (reduce along the days axis). Print the results.
2. Find the **global minimum** and **global maximum** temperature across all stations and days.
3. For each station, find **which measurement** (0–5) had the highest temperature.  
   *Hint:* use `argmax` with the appropriate `dim`.

In [ ]:
# 1. Mean and std corrected temperature per station (reduce along dim=1, the days axis)
station_mean = ??
station_std  = ??

for i in range(4):
    print(f"Station {i}: mean={station_mean[i]:.2f}°C  std={station_std[i]:.2f}°C")

# 2. Global min and max corrected temperatures
global_min = ??
global_max = ??
print(f"\nGlobal min: {global_min:.2f}°C")
print(f"Global max: {global_max:.2f}°C")

# 3. Day with highest temperature per station
# Hint: use `argmax` with the appropriate `dim`
hottest_day = ??
print(f"\nHottest day per station: {hottest_day.tolist()}")


### Solution

In [ ]:
# Solution Task 3

'''
# 1. Mean and std per station (reduce along dim=1, the days axis)
station_mean = temps_corrected.mean(dim=1)
station_std  = temps_corrected.std(dim=1)

for i in range(4):
    print(f"Station {i}: mean={station_mean[i]:.2f}°C  std={station_std[i]:.2f}°C")

# 2. Global min and max
global_min = temps_corrected.min()
global_max = temps_corrected.max()
print(f"\nGlobal min: {global_min:.2f}°C")
print(f"Global max: {global_max:.2f}°C")

# 3. Day with highest temperature per station
hottest_day = temps_corrected.argmax(dim=1)
print(f"\nHottest day per station: {hottest_day.tolist()}")
'''

---
## Task 4: Day/night temperature model (math functions + broadcasting)

Model how temperature varies throughout the day using a sine wave.

1. Create a `(6,)` tensor `hours` with 6 evenly spaced values from 0 to 20 (both ends included) using `torch.linspace`.
2. Compute a daily variation curve: $\text{variation} = 5 \cdot \sin\!\left(\frac{(\text{hours} - 6) \cdot \pi}{12}\right)$  
   This peaks around noon and is coldest around midnight.
3. Take the **mean temperature per station** from Task 3 (shape `(4,)`) and reshape it to `(4, 1)`.
4. **Broadcast-add** the station means `(4, 1)` and the variation `(6,)` to produce a `(4, 6)` modelled temperature matrix.
5. Compute the **mean of the modelled temperatures per station** and verify they are close to the original station means.  
   *Hint:* the sine function averages to roughly zero over a full cycle.

In [ ]:
# 1. Hours of the day
# Create a `(6,)` tensor `hours` with 6 evenly spaced values from 0 to 20 (both ends included) using `torch.linspace`
hours = ??
print(f"Hours: {hours.tolist()}")

# 2. Daily variation curve
variation = ??
print(f"Variation: {[f'{v:.2f}' for v in variation.tolist()]}")

# 3. Reshape station means for broadcasting
station_mean_col = ??

# 4. Broadcast-add: (4,1) + (6,) -> (4,6)
modelled = ??
print(f"\nModelled temperatures shape: {modelled.shape}")
print(modelled)

# 5. Verify: mean of modelled should be close to original station means
modelled_mean = modelled.mean(dim=1)
print(f"\nOriginal station means:  {[f'{v:.2f}' for v in station_mean.tolist()]}")
print(f"Modelled station means:  {[f'{v:.2f}' for v in modelled_mean.tolist()]}")
print(f"Difference:              {[f'{v:.4f}' for v in (modelled_mean - station_mean).tolist()]}")

### Solution

In [ ]:
# Solution Task 4

'''
# 1. Hours of the day
hours = torch.linspace(0, 20, 6)
print(f"Hours: {hours.tolist()}")

# 2. Daily variation curve
variation = 5 * torch.sin((hours - 6) * math.pi / 12)
print(f"Variation: {[f'{v:.2f}' for v in variation.tolist()]}")

# 3. Reshape station means for broadcasting
station_mean_col = station_mean.reshape(4, 1)  # shape (4, 1)

# 4. Broadcast-add: (4,1) + (6,) -> (4,6)
modelled = station_mean_col + variation
print(f"\nModelled temperatures shape: {modelled.shape}")
print(modelled)

# 5. Verify: mean of modelled should be close to original station means
modelled_mean = modelled.mean(dim=1)
print(f"\nOriginal station means:  {[f'{v:.2f}' for v in station_mean.tolist()]}")
print(f"Modelled station means:  {[f'{v:.2f}' for v in modelled_mean.tolist()]}")
print(f"Difference:              {[f'{v:.4f}' for v in (modelled_mean - station_mean).tolist()]}")
'''